# Bank marketing and Term Deposit Prediction

Lien Kaggle : [data_link](https://archive.ics.uci.edu/dataset/222/bank+marketing)

Introduction Paper: [paper_link](https://www.semanticscholar.org/paper/cab86052882d126d43f72108c6cb41b295cc8a9e)


About data

The data is related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be ('yes') or not ('no') subscribed. 

Author :
ATJI Cheick

## Libraries loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import math
from scipy.stats import (pointbiserialr, chi2_contingency)
import statsmodels.api as sm


from xgboost import XGBClassifier


from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier)

from sklearn.metrics import (f1_score, precision_score, recall_score, confusion_matrix,
                            accuracy_score, roc_auc_score, roc_curve, classification_report,
                            auc, ConfusionMatrixDisplay, RocCurveDisplay, get_scorer_names,
                            precision_recall_curve, average_precision_score)

from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold,
                                     cross_val_score,
                                     learning_curve)



from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import (RandomUnderSampler, TomekLinks, NearMiss)

import warnings
import openpyxl

import random
from sklearn.model_selection import StratifiedKFold, learning_curve, train_test_split

warnings.filterwarnings(action = 'ignore')
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)
random.seed(123)

## data loading

In [ ]:
#description type de variables
dictionnaire = {'age':'Age',
                'job':'Type of job, ex: entrepreneur',
                'marital':'Marital status, ex: divorced',
                'education':'education level, ex: high-school',
                'default':'Has credit in default? yes or no',
                'balance':'average yearly balance, ex: 24,000 USD',
                'housing':'has housing loan? yes or no',
                'loan':'has personal loan? yes or no',
                'contact':'cellular or telephone',
                'day':'last contact day of the month',
                'month':'last contact month of the year',
                'duration':'last contact duration in seconds',
                'campaign':'number of contacts performed during this campaign and for this client (numeric, includes last contact)',
                'pdays':'number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)',
                'previous':'number of contacts performed before this campaign and for this client (numeric)',
                'poutcome':'outcome of the previous marketing campaign (categorical: unknown, other,failure,success)',
                'y':'has the client subscribed a term deposit? (binary: yes or no)'
                }


def info(data):

    Information = pd.DataFrame({
        'Variables': data.columns,
        'Type': data.dtypes,
        'Unique_values': data.nunique(),
        'NA_counts': data.isna().sum(),
        'NA_percent%':data.isna().mean().round(4)*100,
        }).reset_index(drop=True)

    Information['Description_des_variables'] = Information['Variables'].map(dictionnaire)
    
    print(f"Taille: {data.shape[0]} - Nb Variables: {data.shape[1]}")
    return Information

In [ ]:
data = pd.read_csv('data/bank.csv', sep=';')

## fonction pour caster les variables

In [ ]:


def conversion(var):

    # conversion des variables type = <objet>
    if (var.dtype == 'object'):
        try:
            
            # conversion en string
            if var.nunique() > 12:
                return var.astype(str)
            
            # conversion en catégorie 
            else:
                return var.astype('category')
        
        except ValueError:
            
            return var
    
    else:
        return var 


In [ ]:
data = data.apply(conversion)

In [ ]:
tableau = info(data)
tableau.to_excel("export.xlsx", index=False, engine="openpyxl")
tableau

- Lister les variables catégorielles et les variables numériques

In [ ]:
#Renommer la variable y
data.rename(columns={"y": "credit_subscribed"}, inplace=True)

# liste variables numériques
numvars = data.select_dtypes(exclude=['category']).columns.tolist()

#liste variables catégorielles
catvars = data.select_dtypes(include=['category']).columns.tolist()

## Visualisation des données 

#### Récodage de certaines  variables

In [ ]:
data['job'] = data['job'].replace(['unemployed', 'self-employed', 'student', 'services', 
                                   'housemaid', 'retired','entrepreneur', 'unknown'], 'others')

data['poutcome'] = data['poutcome'].replace('other','success')

data['education'] = data['education'].replace('unknown','primary')

In [ ]:

def category_distribution(data, var):
    # Validation des données
    if not isinstance(data, pd.DataFrame):
        raise TypeError('data is not a pandas.DataFrame')

    if var not in data.columns or data[var].dtype != 'category':
        raise TypeError(f'{var} not in data or not a <dtype.category>')
    
    # Calcul des fréquences et des pourcentages
    counts = data[var].value_counts()
    percentages = data[var].value_counts(normalize=True) * 100

    # Création du tableau
    distribution_table = pd.DataFrame({
        'Nombre d\'observations': counts,
        'Pourcentage (%)': percentages.round(2)
    })
        # Calcul des totaux
    total_counts = counts.sum()
    total_percentage = percentages.sum()

    # Ajout de la ligne des totaux
    distribution_table.loc['Total'] = [total_counts, total_percentage]
    
    print(f"Repartition des effectifs: {var}")
    
    return distribution_table


In [ ]:
# repartition de la variable d'intérêt
repartition_y = category_distribution(data, 'credit_subscribed')
repartition_y.to_excel("repartition.xlsx", index=True, engine="openpyxl")
repartition_y

In [ ]:
# Visualiser la répartition
plt.figure(figsize=(8, 6))
sns.countplot(x='credit_subscribed', data = data)
plt.title('Répartition de la variable Subscribed')
plt.xlabel('Credit Subscribed (0: No, 1: Yes)')
plt.ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
def plot_categorical_distribution(df, target, features, n_cols=3):
    """
    Affiche la répartition de la variable cible en fonction d'une liste de variables catégorielles
    en utilisant seaborn.countplot.
    
    Paramètres
    ----------
    df : pandas.DataFrame
        Jeu de données.
    target : str
        Nom de la variable catégorielle cible.
    features : list of str
        Liste des variables catégorielles explicatives.
    n_cols : int
        Nombre de colonnes dans la grille d'affichage.
    """
    n_features = len(features)
    n_rows = math.ceil(n_features / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharey=False)

    axes = axes.flatten()

    for ax, feature in zip(axes, features):
        sns.countplot(data=df, x=feature, hue=target, ax=ax)
        ax.set_title(f"Répartition de {target}\nselon {feature}")
        ax.set_ylabel("Nombre d'observations")
        ax.tick_params(axis='x', rotation=45)  # rotation si labels trop longs

    # Supprimer les axes vides si nb variables < n_rows * n_cols
    for ax in axes[n_features:]:
        ax.remove()

    plt.tight_layout()
    plt.show()


In [ ]:
maliste = [x for x in catvars if x not in ['credit_subscribed', 'month']]
plot_categorical_distribution(data, 'credit_subscribed', maliste, n_cols=2)

### Visualisations croisées entre la variable d'intérêt et les variables continues

In [ ]:

# Calcul du nombre de lignes et de colonnes pour la grille
n_vars = len(numvars)
n_cols = 3
n_rows = math.ceil(n_vars / n_cols)

# Création de la grille de sous-graphiques
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows)) # Ajuster la taille si besoin

# Affichage des box plots dans les sous-graphiques
for i, variable in enumerate(numvars):
    row = i // n_cols
    col = i % n_cols
    sns.boxplot(x='credit_subscribed', y=variable, data=data, ax=axes[row, col])
    axes[row, col].set_title(f'Comparaison de {variable} entre les groupes')
    axes[row, col].set_xlabel('Default (0: No, 1: yes)')
    axes[row, col].set_ylabel(variable)

# Ajustement de l'espacement entre les sous-graphiques
plt.tight_layout()

# Affichage de la grille de sous-graphiques
plt.show()

In [ ]:
# Création de la figure avec 2 lignes et 6 colonnes
fig, axes = plt.subplots(4, 2, figsize=(12, 18))

# Liste des colonnes à afficher
columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
           'poutcome']

# Boucle pour créer les graphiques
for i, col in enumerate(columns):
    row = i // 2
    col_index = i % 2

    data[col].value_counts().plot.pie(ax=axes[row, col_index], autopct='%1.1f%%', startangle=90)
    axes[row, col_index].set_title(f"Répartition {col}")
    axes[row, col_index].set_ylabel("")


plt.tight_layout()
plt.show()

## Analyses Statistiques 

### Corrélation entre les variables

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(data[numvars].corr(method='kendall'),
            cmap='Blues', annot=True)
plt.title('Correlations Matrix: method = Kendall\n')
plt.xticks(rotation=45, ha='right')
plt.show()

### Tableau statistiques croisés sur solde de crédit  

In [ ]:
def cross_stat(data:pd.DataFrame, catlist:list, statvars:list):

    """--Docstring--
    fonction pour réaliser des statistiques
    sur des variables croisées.
    
    Args:
         data: le dataframe
         catlist: (list) variables catégorielles
         statvars: (list) variables continues
    
    return a dataframe
    """
    # initialisation d'un tableau vide
    table = pd.DataFrame()
    
    # Création de dictionnaire pour les indexes
    index_mapping = {value: cat for cat in catlist for value in data[cat].unique()}

    
    for var in statvars:

        for cat in catlist:
            
            X = data.groupby(by=cat)[var].agg(min = 'min',
                                              max = 'max',
                                              mean = 'mean',
                                              st_deviation = 'std',
                                              quartile1 = lambda x: x.quantile(0.25),
                                              median = 'median',
                                              quartile3 = lambda x: x.quantile(0.75),
                                              category_size = 'count')

            table = pd.concat([table,X], axis = 0, ignore_index = False)
    
        table.reset_index(names = 'valeurs', inplace = True)
        table['Categories'] = table['valeurs'].map(index_mapping)
        table.set_index(['Categories','valeurs'], inplace = True)
    
        print(f'Tableau Statistiques croisées sur: {var}')
        display(table.round(2))
        print('\n\n')


In [ ]:
cross_stat(data, catvars, ['balance'])

### Test d'associations du khi-deux entre les variables catégorielles et la variable cible 

In [ ]:

def test_khi2(data, target, variables, alpha=0.05):
    """
    Calcule les tests du Chi2 entre une liste de variables catégorielles et une variable cible binaire.

    Parameters:
        data (pd.DataFrame): le DataFrame contenant les données
        target (str): nom de la variable cible (ex: "Default")
        variables (list): liste des variables catégorielles
        alpha (float): seuil de significativité (par défaut 0.05)

    Returns:
        pd.DataFrame: tableau récapitulatif des résultats
    """
    results = []

    for var in variables:
        contingency_table = pd.crosstab(data[var], data[target])
        chi2, p, dof, expected = chi2_contingency(contingency_table)

        decision = "Rejeter H0 (association)" if p < alpha else "Ne pas rejeter H0 (indépendance)"
        results.append({
            "Variable": var,
            "Chi2": chi2,
            "ddl": dof,
            "p-value": p,
            "Décision (alpha={})".format(alpha): decision
                        })

    return pd.DataFrame(results)



- Application du test

In [ ]:
liste = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

khi_deux = test_khi2(data, target = "credit_subscribed", variables = liste)
khi_deux.to_excel("khi_deux.xlsx", index=False, engine="openpyxl")
khi_deux

### Test de corrélation Pearson entre la variable d'intérêt et les variables continues

In [ ]:
def corr_pearson(data, target, variables, alpha=0.05):
    """
    Calcule la corrélation point-bisérial entre une variable binaire et plusieurs variables continues.

    Parameters:
        data (pd.DataFrame): DataFrame contenant les données
        target (str): nom de la variable binaire (ex: "Default")
        variables (list): liste des variables continues
        alpha (float): seuil de significativité (par défaut 0.05)

    Returns:
        pd.DataFrame: tableau récapitulatif avec corrélation, p-value et décision
    """
    results = []

  
    a, b = data[target].unique()
    y = data[target].map({a:0, b:1}).astype('float')

    for var in variables:
        x = data[var]

        # Calcul corrélation point-bisérial
        corr, pval = pointbiserialr(y, x)

        decision = "Rejeter H0 (association)" if pval < alpha else "Ne pas rejeter H0 (indépendance)"
        results.append({
            "Variable": var,
            "Corrélation (r_pb)": corr,
            "p-value": pval,
            "Décision (alpha={})".format(alpha): decision
        })

    return pd.DataFrame(results)


- Application du test de correlation

In [ ]:
correlation_pearson = corr_pearson(data, target = "credit_subscribed", variables = numvars)
correlation_pearson.to_excel("correlation.xlsx", index=False, engine="openpyxl")
correlation_pearson

## Modélisation du Score

### Modèles Statistiques standart

- Préparation des données

In [ ]:
# Variable d'intérêt
y = data['credit_subscribed'].map({'yes':1,'no':0}).astype('int')

# Variables explicatives
quantvars = [x for x in numvars if x not in ['duration', 'previous', 'balance', 'day', 'pdays']]
categvars =  [x for x in catvars if x not in ['education','marital','job', 'month', 'contact','credit_subscribed', 'default']]
variables = quantvars + categvars

# Matrices des regresseurs
X = pd.get_dummies(data[variables],
                   columns=categvars,
                   drop_first=True).astype(int)

X_cst = sm.add_constant(X, prepend=False)

#### Regression logistique

In [ ]:
model_logistic = sm.Logit(y,X_cst)
results = model_logistic.fit(methode = 'newton')
results.summary2()

#### Les modèles GLM: 

##### Binomial family

In [ ]:
binomial = sm.families.Binomial()
model_glm = sm.GLM(y, X_cst, family = binomial)
results = model_glm.fit()
results.summary2()

##### complementary log-Log (Clog-log) family

In [ ]:
# Complementary log-log
cloglog = sm.families.Binomial(link=sm.families.links.cloglog())
model_cloglog = sm.GLM(y, X_cst, family = cloglog)
results = model_cloglog.fit()
results.summary2()
#np.exp(results.params)

### Machine Learning

#### Decoupage des données en train/test 

- selection des variables (features)

In [ ]:
X_cats = ['job', 'marital', 'education','housing', 'loan','poutcome', 'contact']
X_nums = ['age', 'balance', 'day', 'duration', 'campaign', 'previous']

In [ ]:
#Séparer les données en features (X) et target (y)
data_sorted = data.sort_values(by='balance', ignore_index=True)
features = data_sorted[X_cats + X_nums]
target = data_sorted["credit_subscribed"].map({'no':0, 'yes':1}).astype('int')

#Diviser les données en train et test

X_train, X_test, y_train, y_test = train_test_split(
                                                    features, target, 
                                                    test_size=0.2, 
                                                    random_state=42,
                                                    shuffle = True, 
                                                    stratify=target
                                                    )

print("Taille de l'ensemble d'entraînement :", len(X_train))
print("Taille de l'ensemble de test :", len(X_test))

- Surechantillonnage du train set par la methode SMOTENC

In [ ]:
# Déterminer l'indice des colonnes catégorielles dans le trainset 
position = [X_train.columns.get_loc(x) for x in X_cats]

# Création de l'instance SMOTE
smote = SMOTENC(categorical_features=position, random_state=42)

# Réechantillonnage (fitting)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

- Sousechantillonnage du train set 

In [ ]:
rd_sampling = RandomUnderSampler(random_state=42) # reduce majority class
tk_sampling = TomekLinks() # removing noises from dataset
nm_sampling = NearMiss(n_neighbors=5) # intelligent Selecting

X = pd.get_dummies(X_train, columns = X_cats, drop_first=True)

X_sampled, y_sampled = nm_sampling.fit_resample(X, y_train)

# Retrouver les colonnes d'origine
X_train.sort_values(by='balance', inplace = True, ignore_index=True)
X_sampled.sort_values(by = 'balance', inplace = True, ignore_index=True)
merged = pd.merge(X_sampled, X_train, left_index=True, right_index=True, how="inner")
merged.rename(columns=lambda x: x[:-2] if x.endswith("_y") else x, inplace = True)
X_sampled = merged[X_cats + X_nums]

print(f"{X_sampled.shape}\n\n{y_sampled.value_counts()}")

- Affichage des sets

In [ ]:
fig, ax=plt.subplots(1,2,figsize=(11,5), sharex=False, sharey=False)
ax1, ax3= ax
ax1.scatter(X_train.iloc[:,6],X_train.iloc[:,7], c=y_train, alpha=0.8, lw=3)
ax1.set_title('Train Data (80%)')
ax3.scatter(X_test.iloc[:,6],X_test.iloc[:,7], c=y_test, alpha=0.8, lw=3)
ax3.set_title('Test Data (20%)')
plt.show()

### Création du pipeline pour enchainer les opérations

In [ ]:
def create_pipeline(model: any, X_nums, X_cats):

    # Transformer : scale les numériques et encode les catégorielles
    transformer = ColumnTransformer([
                                        ("num", StandardScaler(), X_nums),
                                        ("cat", OneHotEncoder(drop='first', handle_unknown="ignore"), 
                                         X_cats)
                                    ])

    # Build pipeline
    pipeline = Pipeline([
                            ("preprocessing", transformer),
                            ("classification", model)
                        ])

    return pipeline


### Création des Modèles

In [ ]:
# Number of folds = 10 for unbalanced data
stratfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
logistic_classifier = create_pipeline(LogisticRegressionCV(class_weight = "balanced",
                                                                      cv = stratfold,
                                                                      random_state = 42),
                                                            X_nums,X_cats)
                                  
xgboost_classifier = create_pipeline(XGBClassifier(random_state = 42), 
                                     X_nums,X_cats)

adaboost_classifier = create_pipeline(AdaBoostClassifier(n_estimators=100, random_state = 42),
                                      X_nums,X_cats)
                                      

grdboost_classifier = create_pipeline(GradientBoostingClassifier(n_estimators=100, random_state = 42),
                                      X_nums,X_cats)
                                      

randomforest_classifier = create_pipeline(RandomForestClassifier(class_weight = "balanced",
                                                                 n_estimators = 150,
                                                                 random_state = 42),
                                          X_nums, X_cats)

### Entrainement

#### Fonction pour entrainer les modèles

In [ ]:
def running_model(model: any, name_model: str, X:pd.DataFrame, y: pd.Series, threshold: float = 0.5):

    # Méthodes de splitting pour variable déséquilibrée
    stratfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Entraînement final du modèle
    model.fit(X, y)

    # Prédictions probabilistes
    y_prob = model.predict_proba(X_test)[:, 1]

    # seuil de probabilité pour la classification
    y_pred = (y_prob >= threshold).astype(int)


    # Classification report
    print(classification_report(y_test, y_pred))

    confusion = confusion_matrix(y_test, y_pred, labels=[1,0])
    matrix=ConfusionMatrixDisplay(np.transpose(confusion), display_labels=['Yes: 1','No: 0'])


    train_size,train_score, val_score = learning_curve(model,
                                                        X, y.to_numpy().ravel(),
                                                        train_sizes=np.linspace(0.1, 1,10),
                                                        cv=stratfold,shuffle=True,random_state=0,
                                                        return_times=False)

    # Tracé des graphes 

    fig = plt.figure(figsize=(11, 9))
    gs = fig.add_gridspec(2, 2)  # 2 lignes, 2 colonnes

    ax1 = fig.add_subplot(gs[0, 0])  # ligne 0, col 0
    ax2 = fig.add_subplot(gs[0, 1])  # ligne 0, col 1
    ax3 = fig.add_subplot(gs[1, :])  # ligne 1, toutes les colonnes

    # Matrix de Confusion
    matrix.plot(cmap='Blues', ax=ax1, colorbar=False)
    ax1.text(-0.25,.9,'False Negative',c='blue')
    ax1.text(0.75,0.9,'True Negative',c='white')
    ax1.text(-.25,-.15,'True Positive', c='blue')
    ax1.text(0.75,-.15,'False Positive',c='blue')
    ax1.set_ylabel('Model predictions')
    ax1.set_xlabel('Actual Values')
    ax1.xaxis.set_ticks_position('top')
    ax1.xaxis.set_label_position('top')
    ax1.set_title(f'Confusion Matrix for: {name_model}')

    # Courbe ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    ax3.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
    ax3.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

    ax3.set_xlim([0.0, 1.0])
    ax3.set_ylim([0.0, 1.05])
    ax3.set_xlabel('Taux de faux positifs')
    ax3.set_ylabel('Taux de vrais positifs')
    ax3.set_title(f'Courbe ROC - {name_model}')
    ax3.legend(loc="lower right")

    ax2.plot(train_size,train_score.mean(axis=1),c='b',lw=2,label='Train_score')
    ax2.plot(train_size,val_score.mean(axis=1),c='r',lw=2,label='validation_score')
    ax2.set_ylabel('Accuracy')
    ax2.set_xlabel('Training Set Size')
    ax2.set_title(f"Train vs Validation Score: {name_model}")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    return model

#### Logistic Classifier

In [ ]:
logistic = running_model(logistic_classifier, "Logistic Classifier",X_smote, y_smote, threshold = 0.5)

- Courbe Précision-Rappel

In [ ]:
# Fonction pour la courbe précision rappel:

def curve_precison_recall(model:any, name:str) -> any:    
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
    avg_precision = average_precision_score(y_test, y_pred_proba)

    plt.figure(figsize=(8,6))
    plt.plot(recall, precision, label=f'PR Curve (AP = {avg_precision:.2f})')
    plt.xlabel('Rappel')
    plt.ylabel('Précision')
    plt.title('Courbe Précision-Rappel - Logistic Classifier')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
y_pred_proba = logistic.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

plt.figure(figsize=(8,6))
plt.plot(recall, precision, label=f'PR Curve (AP = {avg_precision:.2f})')
plt.xlabel('Rappel')
plt.ylabel('Précision')
plt.title('Courbe Précision-Rappel - Logistic Classifier')
plt.legend()
plt.grid(True)
plt.show()


#### AdaBoostclassifier

In [ ]:
adaboost = running_model(adaboost_classifier, 'AdaBoosting', X_smote, y_smote, threshold = 0.5)

- Courbe Précision Rappel : Adaboost

In [ ]:
y_pred_proba = adaboost.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

plt.figure(figsize=(8,6))
plt.plot(recall, precision, label=f'PR Curve (AP = {avg_precision:.2f})')
plt.xlabel('Rappel')
plt.ylabel('Précision')
plt.title('Courbe Précision-Rappel - Logistic Classifier')
plt.legend()
plt.grid(True)
plt.show()


#### GradientBoostingClassifier

In [ ]:
running_model(grdboost_classifier,'GradientBoosting', X_smote, y_smote, threshold = 0.5)

#### XGBoostClassifier

In [ ]:
running_model(xgboost_classifier,'XgBoost', X_smote, y_smote, threshold=0.5)

#### Random Forest

In [ ]:
running_model(randomforest_classifier, "Random_Forest", X_smote, y_smote, threshold = 0.5)